In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from zipfile import ZipFile

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

from time import sleep, time

import os



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'US FDIC' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 

    # 'US FDIC 1': '', 

    # 'US FDIC 2': '', 

    # 'US FDIC 3': '', 

    # 'US FDIC 4': '', 

    # 'US FDIC 5': '', 

    # 'US FDIC 6': '', 

    # 'US FDIC 7': '', 

    # 'US FDIC 8': 'https://www5.fdic.gov/sod', 

    'US FDIC 9': 'https://www.fdic.gov/regulations/resources/minority/mdi.html'

            }



#Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



df_institutions2 = None

columns_US_FDIC_9 = ['NAME','CITY','STATE','EST_DATE','CERT','CLASS','REGULATOR','MINORITY_STATUS_Alpha','MINORITY_STATUS','FDIC_REGION','TOTAL_ASSETS']

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def click_element(XPATH, Msg=None):

    for time in range(10):

        try:

            driver.find_element(By.XPATH, XPATH).click()

            if Msg!=None:

                print(f"[INFO] : - {Msg}")

            break

        except:

            print(f"[ERROR] : - Retrying {time+1}/10 to click element in this page: {XPATH} ")

            sleep(1)

    else:

        raise Exception('[ERROR] : Failed to get presence for this element in this page:')



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)}")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/{wait_time} s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def unzip(source_path, output_path):

    with ZipFile(source_path, 'r') as zip_:

        zip_.extractall(output_path)



def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def scroll_to_bottom(driver):
    # Get scroll height
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        # Scroll down to the bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load the page
        sleep(2)

        # Calculate new scroll height and compare with last scroll height
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):



    driver.get(regdict[reg])

    print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} ")



    if reg =='US FDIC 8':



        click_element('//*[@id="SubmitButton"]', "< Summary of Deposits >: --> Click 'continue' Button 1/3")

        click_element('//*[@id="submit1"]', "< County Selection >: --> Click 'continue' Button 2/3")

        click_element('//*[@value="Run Report"]', "< Selected Criteria Summary >: --> Click 'Run Report' Button 3/3")

        sleep(4)

        try :

            feedback_bannier = driver.find_element(By.XPATH,f'//*[@id="fsrinvite"]').is_displayed()

        except:

            feedback_bannier = False

        if feedback_bannier :

            print(f"[INFO] : - Decline feedback bannier ")

            driver.find_element(By.XPATH,'//*[@id="decline"]').click()

        

        print(f"[INFO] : - DataFrame {file} | containe = {df.shape}")

        for index, row in df.iterrows() :

            sqldict['Name'].append(str(row['NAMEFULL']))

            sqldict['Address_1'].append( str(row['ADDRESS']))

            sqldict['InternalID_1'].append(str(row['CERT'])) # RSSDID , BRNUM, CERT, UNINUMBR, CERT, RSSDHCR, STNUMBR, STCNTYBR, CSABR

            sqldict['InternalID_1_type'].append('CERT')

            sqldict['InternalID_2'].append(str(row['RSSDID']))

            sqldict['InternalID_2_type'].append('RSSDID')

            sqldict['InternalID_3'].append(str(row['RSSDHCR']))

            sqldict['InternalID_3_type'].append('RSSDHCR')

            sqldict['RegulationDate'].append(str(row['SIMS_ESTABLISHED_DATE'])) # SIMS_ESTABLISHED_DATE, YEAR

            sqldict['City'].append(str(row['CITY']))

            sqldict['Zip'].append(str(row['ZIP']))

            sqldict['Cntry'].append(reg.split(' ')[0])

            sqldict['RegulationType'].append('Regulated')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

            #sqldict['Typology'].append(regdict[reg])

  

    elif reg =='US FDIC 9':

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # ul = soup.find('ul', {'class': 'glossary'})

        # a_s = ul.find_all('a', href=True)
        sleep(10)
        
        scroll_to_bottom(driver)
        
        sleep(5)
        data_file_module = soup.find('div',class_='paragraph')
        sleep(5)
        if data_file_module:
            data_file_ul = data_file_module.find('ul')
        else:
            sleep(5)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            sleep(5)
            data_file_module = soup.find('div',class_='paragraph')
            sleep(5)
            data_file_ul = data_file_module.find('ul')
        if data_file_ul:
            sleep(5)
            data_file = data_file_ul.find('li').find('a')['href']
        else:
            sleep(5)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            sleep(5)
            data_file_module = soup.find('div',class_='paragraph')
            sleep(5)
            data_file_ul = data_file_module.find('ul')
            sleep(5)
            data_file = data_file_ul.find('li').find('a')['href']
        driver.get('https://www.fdic.gov/'+data_file)

        

        file = check_dowload_files(tempfolder, "Excel", 30 )

        filePath = os.path.join(tempfolder, file)

        df = pd.read_excel(filePath, dtype=str)

        df.columns = columns_US_FDIC_9

        df = df[1:-34]

        df = df.fillna("")

        print(f"[INFO] : - DataFrame {file} | containe = {df.shape}")



        for index, row in df.iterrows() :

            sqldict['Name'].append(str(row['NAME']))

            sqldict['Address_1'].append( str(row['FDIC_REGION']))

            sqldict['InternalID_1'].append(str(row['CERT']))

            sqldict['InternalID_1_type'].append('CERT')

            # sqldict['InternalID_2'].append(str(row['EST_DATE']))

            # sqldict['InternalID_2_type'].append('EST_DATE')

            sqldict['ListName'].append('Minority Depository Institutions List')

            sqldict['City'].append(str(row['CITY']))

            sqldict['Cntry'].append(reg.split(' ')[0])

            sqldict['RegulationType'].append('Regulated')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

            sqldict['Typology'].append(str(row['REGULATOR']))



    # else:

    #     if df_institutions2 is None:

    #         driver.get('https://www5.fdic.gov/idasp/Institutions2.zip')

    #         sleep(4)

    #         zip_file = check_dowload_files(tempfolder, "Zip", 30 )

    #         zip_file = os.path.join(tempfolder, zip_file)

    #         unzip(zip_file, tempfolder)

    #         os.remove(zip_file)



    #         file = check_dowload_files(tempfolder, "CSV", 30 )

    #         filePath = os.path.join(tempfolder, file)

    #         df = pd.read_csv(filePath, dtype=str, encoding ='ISO-8859-1')

    #         df = df.fillna("")

    #         print(f"[INFO] : - DataFrame {file} | containe = {df.shape}")



    #         for index, row in df.iterrows() :

    #             sqldict['Name'].append(str(row['NAMEFULL']))

    #             sqldict['Address_1'].append( str(row['ADDRESS']))

    #             sqldict['InternalID_1'].append(str(row['CERT'])) 

    #             sqldict['InternalID_1_type'].append('CERT')

    #             sqldict['InternalID_2'].append(str(row['RSSDID']))

    #             sqldict['InternalID_2_type'].append('RSSDID')

    #             sqldict['InternalID_3'].append(str(row['RSSDHCR']))

    #             sqldict['InternalID_3_type'].append('RSSDHCR')

    #             sqldict['RegulationDate'].append(str(row['SIMS_ESTABLISHED_DATE']))

    #             sqldict['City'].append(str(row['CITY']))

    #             sqldict['Zip'].append(str(row['ZIP']))

    #             sqldict['Cntry'].append(reg.split(' ')[0])

    #             sqldict['RegulationType'].append('Regulated')

    #             sqldict['ListProcessDate'].append(processdate)

    #             sqldict['RegCtry'].append(reg.split(' ')[0]) 

    #             sqldict['RegCode'].append(reg.split(' ')[1])

    #             sqldict['ListCode'].append(reg.split(' ')[-1])

    #             #sqldict['Typology'].append(regdict[reg])



    sqldict = bourange_same_length_array(sqldict)

    for del_file in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, del_file))



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

Running US FDIC Web Scraping Tool v.1.1
[INFO] : Working 1/1 | US FDIC 9 
[INFO] : - Download Excel file ... (wait 0/30 s)
[INFO] : - Excel file = ['fourth-quarter-2024-xlsx.xlsx']
[INFO] : - DataFrame fourth-quarter-2024-xlsx.xlsx | containe = (155, 11)


C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_25640\3367993517.py:495: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('list9_v.csv')